In [113]:
import pandas as pd
import numpy as np

In [114]:
data_df = pd.read_csv("yolo_data/all_csv_files/combined_dataset_info_v5.csv")

In [174]:
train_or_test = "train"

In [175]:
CG_data_df = data_df[(data_df["dataset"] == "CG") & (data_df["train_or_test"] == train_or_test)]

In [176]:
CG_data_df

,dataset,patient_ID,file_name,nodule_type,train_or_test
8493,CG,1268387F084C766E5899EDF651940EAF4EB8E429,3c53bbc2f24e91b75d3db806b46dd9e3_left,single,train
8494,CG,1268387F084C766E5899EDF651940EAF4EB8E429,19b7fce9c1bfea6b689191953a6f7326_right,single,train
8495,CG,1268387F084C766E5899EDF651940EAF4EB8E429,013d9a8640d9d25e766894901746fe09_left,single,train
8496,CG,1268387F084C766E5899EDF651940EAF4EB8E429,013d9a8640d9d25e766894901746fe09_right,single,train
8497,CG,1268387F084C766E5899EDF651940EAF4EB8E429,3c53bbc2f24e91b75d3db806b46dd9e3_right,multiple,train
...,...,...,...,...,...
20631,CG,E834023AF744DF427D3FD45C861019952A271C55,f0c53427e4020567b9927aa86f1c989e_left,none,train
20632,CG,83CBDF3C042756B058D4D8BB4A13E866B368334C,c5fd5d8121a6179cb8d2a383161594b2_right,none,train
20633,CG,83CBDF3C042756B058D4D8BB4A13E866B368334C,09e11b2274006362a2e1b9e0f3009a37_left,none,train
20634,CG,83CBDF3C042756B058D4D8BB4A13E866B368334C,09e11b2274006362a2e1b9e0f3009a37_right,none,train


In [177]:
print("single nodule image :", len(CG_data_df[CG_data_df["nodule_type"] == "single"]))
print("multiple nodule image :", len(CG_data_df[CG_data_df["nodule_type"] == "multiple"]))
print("none nodule image :", len(CG_data_df[CG_data_df["nodule_type"] == "none"]))

single nodule image : 5545
multiple nodule image : 354
none nodule image : 3889


In [178]:
print("single nodule patient :", len(np.unique(CG_data_df[CG_data_df["nodule_type"] == "single"]["patient_ID"])))
print("multiple nodule patient :", len(np.unique(CG_data_df[CG_data_df["nodule_type"] == "multiple"]["patient_ID"])))
print("none nodule patient :", len(np.unique(CG_data_df[CG_data_df["nodule_type"] == "none"]["patient_ID"])))

single nodule patient : 1646
multiple nodule patient : 200
none nodule patient : 917


## 有結節影像：平均每張結節數

In [179]:
label_dir = 'yolo_data/all_data_nodule_normal_cut_od/labels'

def count_nodules(fname):
    with open(f'{label_dir}/{fname}.txt') as f:
        return sum(1 for line in f if line.strip())

CG_data_df = CG_data_df.copy()
CG_data_df['nodule_count'] = CG_data_df['file_name'].apply(count_nodules)

nodule_imgs = CG_data_df[CG_data_df['nodule_type'] != 'none']

mean_n = nodule_imgs['nodule_count'].mean()
print(f'有結節影像數: {len(nodule_imgs)}')
print(f'平均每張結節數: {mean_n:.2f}')
print()
print('結節數分布:')
dist = nodule_imgs['nodule_count'].value_counts().sort_index()
dist_df = pd.DataFrame({'結節數': dist.index, '影像張數': dist.values,
                         '比例 (%)': (dist.values / dist.values.sum() * 100).round(1)})
display(dist_df)

print("total nodule : ", CG_data_df['nodule_count'].sum())

有結節影像數: 5899
平均每張結節數: 1.23

結節數分布:


,結節數,影像張數,比例 (%)
0,1,4853,82.3
1,2,804,13.6
2,3,179,3.0
3,4,41,0.7
4,5,14,0.2
5,6,4,0.1
6,7,4,0.1


total nodule :  7284


In [180]:
print("total image :", len(CG_data_df))
print("total patient :", len(np.unique(CG_data_df["patient_ID"])))

total image : 9788
total patient : 2575


# CG 病歷報告統計分析（all_reports）

In [181]:
import pandas as pd
import numpy as np
import os

# ── 讀取所有病歷報告 ──────────────────────────────────────────────
BASE = '../thyroid_old/data/CG_data/all_reports'

dfs = []
for f in sorted(os.listdir(BASE)):
    if not f.endswith('.xlsx'):
        continue
    path = os.path.join(BASE, f)
    try:
        df = pd.read_excel(path, engine='calamine')
    except Exception:
        df = pd.read_excel(path, engine='openpyxl')
    dfs.append(df)

all_df = pd.concat(dfs, ignore_index=True)

# ── 篩選 CG test 病人 ─────────────────────────────────────────────
csv_df = pd.read_csv('yolo_data/all_csv_files/combined_dataset_info_v5.csv')
cg_test_patients = csv_df[
    (csv_df['dataset'] == 'CG') & (csv_df['train_or_test'] == train_or_test)
]['patient_ID'].unique()

filtered = all_df[all_df['IDCODE'].isin(cg_test_patients)].copy()

# EDATE 格式不一，統一轉 datetime
filtered['exam_date'] = pd.to_datetime(filtered['EDATE'], format='mixed', dayfirst=False)

# 每位病人取最早一筆（age 以第一次檢查為準）
patient_df = (filtered
              .sort_values('exam_date')
              .drop_duplicates(subset='IDCODE', keep='first')
              .reset_index(drop=True))

print(f'CG test 病人數 : {len(cg_test_patients)}')
print(f'對回報告後 (dedup): {len(patient_df)} 位病人')

CG test 病人數 : 2575
對回報告後 (dedup): 2575 位病人


## 1. 資料時間範圍

In [182]:
date_min = patient_df['exam_date'].min()
date_max = patient_df['exam_date'].max()

print(f'資料時間範圍: {date_min.strftime("%Y-%m-%d")} ~ {date_max.strftime("%Y-%m-%d")}')
print(f'共橫跨 {(date_max - date_min).days} 天')

資料時間範圍: 1970-01-01 ~ 2022-12-29
共橫跨 19354 天


## 2. 性別分布

In [183]:
sex_counts = patient_df['SEX'].value_counts()
sex_pct = patient_df['SEX'].value_counts(normalize=True) * 100

sex_summary = pd.DataFrame({
    '人數': sex_counts,
    '比例 (%)': sex_pct.round(1)
})
sex_summary.index.name = '性別'
sex_summary.index = sex_summary.index.map({'F': '女 (F)', 'M': '男 (M)'})

print('性別分布:')
display(sex_summary)
print(f'\n總計: {sex_counts.sum()} 人')

性別分布:


,人數,比例 (%)
性別,,
女 (F),1989,77.2
男 (M),586,22.8



總計: 2575 人


In [184]:
patient_df[(patient_df["SEX"]!="M") & (patient_df["SEX"]!="F")]

,LOC,DTYM,IDCODE,EDATE,ITEM,ABSTRACT,DRNO,ODRNO,GRAPHNO,REPORT01,...,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Manufacturer,ManufacturerModelName,positive,exam_date


## 3. 年齡（平均 ± SD）

In [185]:
age = patient_df['age'].dropna()
# age = age.drop(age[age <= 1].index)  # 去除年齡 <= 1 的異常值
age_mean = age.mean()
age_sd   = age.std()
age_min  = age.min()
age_max  = age.max()
age_med  = age.median()

print(f'年齡（平均 ± SD）: {age_mean:.1f} ± {age_sd:.1f} 歲')
print(f'中位數: {age_med:.1f} 歲')
print(f'範圍: {age_min:.1f} ~ {age_max:.1f} 歲')
print(f'有效人數: {len(age)}  缺值: {patient_df["age"].isna().sum()}')

年齡（平均 ± SD）: 47.6 ± 15.6 歲
中位數: 47.8 歲
範圍: 0.0 ~ 103.3 歲
有效人數: 2575  缺值: 0


In [186]:
bins = list(range(0, 120, 10))
labels = [f'{b}–{b+9}' for b in bins[:-1]]

age_group = pd.cut(age, bins=bins, labels=labels, right=False)
age_group_counts = age_group.value_counts().sort_index()
age_group_pct = age_group_counts / age_group_counts.sum() * 100

age_group_df = pd.DataFrame({
    '人數': age_group_counts,
    '比例 (%)': age_group_pct.round(1),
})
age_group_df.index.name = '年齡層'

print('年齡分層（每 10 年）:')
display(age_group_df)
print(f'\n總計: {age_group_counts.sum()} 人')

年齡分層（每 10 年）:


,人數,比例 (%)
年齡層,,
0–9,2,0.1
10–19,72,2.8
20–29,332,12.9
30–39,446,17.3
40–49,553,21.5
50–59,571,22.2
60–69,411,16.0
70–79,154,6.0
80–89,30,1.2



總計: 2575 人


## 4. 綜合摘要表

In [187]:
summary = {
    '病人總數':       len(patient_df),
    '資料起始日':     date_min.strftime('%Y-%m-%d'),
    '資料結束日':     date_max.strftime('%Y-%m-%d'),
    '女性人數 (%)':   f"{sex_counts.get('F', 0)} ({sex_pct.get('F', 0):.1f}%)",
    '男性人數 (%)':   f"{sex_counts.get('M', 0)} ({sex_pct.get('M', 0):.1f}%)",
    '年齡平均 ± SD':  f'{age_mean:.1f} ± {age_sd:.1f}',
    '年齡中位數':     f'{age_med:.1f}',
    '年齡範圍':       f'{age_min:.1f} ~ {age_max:.1f}',
}

summary_df = pd.DataFrame(summary.items(), columns=['項目', '數值'])
display(summary_df)

,項目,數值
0,病人總數,2575
1,資料起始日,1970-01-01
2,資料結束日,2022-12-29
3,女性人數 (%),1989 (77.2%)
4,男性人數 (%),586 (22.8%)
5,年齡平均 ± SD,47.6 ± 15.6
6,年齡中位數,47.8
7,年齡範圍,0.0 ~ 103.3


# 分層分析（Subgroup Analysis）

In [188]:
import cv2, re

label_dir = 'yolo_data/all_data_nodule_normal_cut_od/labels'
image_dir = 'yolo_data/all_data_nodule_normal_cut_od/images'

csv_df = pd.read_csv('yolo_data/all_csv_files/combined_dataset_info_v5.csv')
cg = csv_df[(csv_df['dataset'] == 'CG') & (csv_df['train_or_test'] == train_or_test)].copy()
nodule_cg = cg[cg['nodule_type'] != 'none'].copy()

# 從 label 檔逐顆結節建立 DataFrame
records = []
for _, row in nodule_cg.iterrows():
    fname = row['file_name']
    img_path = f'{image_dir}/{fname}.png'
    if not os.path.exists(img_path):
        img_path = f'{image_dir}/{fname}.jpg'
    img = cv2.imread(img_path)
    if img is None:
        continue
    H, W = img.shape[:2]
    with open(f'{label_dir}/{fname}.txt') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            _, cx, cy, w, h = int(parts[0]), float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
            records.append({
                'file_name':     fname,
                'patient_ID':    row['patient_ID'],
                'nodule_type':   row['nodule_type'],
                'train_or_test': row['train_or_test'],
                'cx': cx, 'cy': cy,
                'w_px': w * W, 'h_px': h * H,
                'max_dim_px': max(w * W, h * H),
            })

nodule_df = pd.DataFrame(records)
print(f'CG test 結節 instances 總數: {len(nodule_df)}')

CG test 結節 instances 總數: 7284


In [189]:
import pydicom
import subprocess
import re
from pathlib import Path

DCM_ROOT = '../thyroid_old/data/CG_data/all_data'

# 取得所有需要查找的 DICOM base hash
nodule_df['dcm_hash'] = nodule_df['file_name'].str.replace(r'_(left|right)$', '', regex=True)
unique_hashes = nodule_df['dcm_hash'].unique()
print(f'需要查找 {len(unique_hashes)} 個 DICOM 檔案')

# 一次性建立 hash → path 索引
result = subprocess.run(['find', DCM_ROOT, '-name', '*.dcm'], capture_output=True, text=True)
all_dcm = result.stdout.strip().split('\n')
hash_to_path = {Path(p).stem: p for p in all_dcm if p}
print(f'索引建立完成，共 {len(hash_to_path)} 個 DICOM 檔案')

# 讀取 pixel spacing（mm/px）
def get_pixel_spacing(dcm_path):
    """回傳 (x方向mm/px, y方向mm/px)，即 (col, row)"""
    try:
        ds = pydicom.dcmread(dcm_path, stop_before_pixels=True)
        if hasattr(ds, 'PixelSpacing'):
            ps = ds.PixelSpacing
            return float(ps[1]), float(ps[0])   # PixelSpacing = [row, col]
        if hasattr(ds, 'SequenceOfUltrasoundRegions'):
            for reg in ds.SequenceOfUltrasoundRegions:
                pdx = getattr(reg, 'PhysicalDeltaX', None)   # cm/px
                pdy = getattr(reg, 'PhysicalDeltaY', None)
                if pdx is not None and pdy is not None:
                    return abs(float(pdx)) * 10, abs(float(pdy)) * 10  # cm→mm
    except Exception:
        pass
    return None, None

spacing_map = {}
not_found = []
for h in unique_hashes:
    if h in hash_to_path:
        spacing_map[h] = get_pixel_spacing(hash_to_path[h])
    else:
        not_found.append(h)
        spacing_map[h] = (None, None)

found = sum(1 for v in spacing_map.values() if v[0] is not None)
print(f'成功讀取 pixel spacing: {found}/{len(unique_hashes)}')
if not_found:
    print(f'找不到 DICOM 檔: {len(not_found)}')

# 將 pixel spacing 和 mm 大小加入 nodule_df
nodule_df['ps_x_mm'] = nodule_df['dcm_hash'].map(lambda h: spacing_map[h][0])  # col方向 mm/px (寬)
nodule_df['ps_y_mm'] = nodule_df['dcm_hash'].map(lambda h: spacing_map[h][1])  # row方向 mm/px (高)
nodule_df['w_mm'] = nodule_df['w_px'] * nodule_df['ps_x_mm']
nodule_df['h_mm'] = nodule_df['h_px'] * nodule_df['ps_y_mm']
nodule_df['max_dim_mm'] = nodule_df[['w_mm', 'h_mm']].max(axis=1)

has_spacing = nodule_df['max_dim_mm'].notna().sum()
print(f'\n有 pixel spacing 的結節數: {has_spacing}/{len(nodule_df)}')
print(f'\n結節實際大小統計（mm）:')
display(nodule_df[['w_mm', 'h_mm', 'max_dim_mm']].describe().round(2))

需要查找 3954 個 DICOM 檔案
索引建立完成，共 37187 個 DICOM 檔案
成功讀取 pixel spacing: 3900/3954

有 pixel spacing 的結節數: 7174/7284

結節實際大小統計（mm）:


,w_mm,h_mm,max_dim_mm
count,7174.00,7174.00,7174.00
mean,11.63,7.78,11.69
std,8.51,5.56,8.51
min,1.36,1.36,1.36
25%,6.00,4.00,6.03
50%,8.86,6.03,8.96
75%,14.34,9.76,14.40
max,68.17,49.86,68.17


## 1. 結節大小分層（依 max dimension 毫米分三群）

> 使用各影像 DICOM 的 Pixel Spacing（`SequenceOfUltrasoundRegions.PhysicalDeltaX/Y`），將 bounding box 像素換算為實際大小（mm），再以三分位切點分群。

In [192]:
# 只用有 pixel spacing 的結節做分層
nodule_mm = nodule_df[nodule_df['max_dim_mm'].notna()].copy()
no_spacing = nodule_df[nodule_df['max_dim_mm'].isna()]
print(f'有 pixel spacing（可換算 mm）: {len(nodule_mm)} 顆')
print(f'無 pixel spacing（排除）     : {len(no_spacing)} 顆\n')

# q33 = nodule_mm['max_dim_mm'].quantile(1/3)
# q67 = nodule_mm['max_dim_mm'].quantile(2/3)
q33 = 6.9
q67 = 12.6
nodule_mm['size_group'] = pd.cut(
    nodule_mm['max_dim_mm'],
    bins=[0, q33, q67, np.inf],
    labels=[f'小 (<{q33:.1f}mm)', f'中 ({q33:.1f}–{q67:.1f}mm)', f'大 (≥{q67:.1f}mm)']
)

# 同步更新 nodule_df 的 size_group（供後續交叉表使用）
nodule_df = nodule_df.merge(
    nodule_mm[['file_name', 'cx', 'size_group']],
    on=['file_name', 'cx'], how='left'
)

size_stats = nodule_mm.groupby('size_group', observed=True).agg(
    結節數=('max_dim_mm', 'count'),
    平均最大邊長_mm=('max_dim_mm', 'mean'),
    中位最大邊長_mm=('max_dim_mm', 'median'),
    最小_mm=('max_dim_mm', 'min'),
    最大_mm=('max_dim_mm', 'max'),
    平均寬_mm=('w_mm', 'mean'),
    平均高_mm=('h_mm', 'mean'),
).round(2)
size_stats['比例 (%)'] = (size_stats['結節數'] / size_stats['結節數'].sum() * 100).round(1)

print(f'三分位切點: {q33:.2f} mm / {q67:.2f} mm\n')
display(size_stats)

有 pixel spacing（可換算 mm）: 7202 顆
無 pixel spacing（排除）     : 110 顆

三分位切點: 6.90 mm / 12.60 mm



,結節數,平均最大邊長_mm,中位最大邊長_mm,最小_mm,最大_mm,平均寬_mm,平均高_mm,比例 (%)
size_group,,,,,,,,
小 (<6.9mm),2456,5.04,5.12,1.36,6.89,5.00,3.63,34.1
中 (6.9–12.6mm),2475,9.34,9.13,6.91,12.58,9.26,6.47,34.4
大 (≥12.6mm),2271,21.38,18.24,12.60,68.17,21.30,13.66,31.5


## 2. 結節位置分層（左葉 / 右葉）與 View（Saggital / Transverse）

In [170]:
info_all = pd.read_excel('../thyroid_old/data/CG_data/all_csv_files/data_info_nodule.xlsx')
info_all['position'] = info_all['position'].str.strip().replace('lright', 'right')
info_all['view'] = info_all['view'].str.strip()

# 只保留 CG test 的影像
test_files = csv_df[(csv_df['dataset']=='CG') & (csv_df['train_or_test']==train_or_test)]['file_name']
info = info_all[info_all['file_name'].isin(test_files)].copy()

# ── 位置分布 ────────────────────────────────────────────────────
pos_counts = info['position'].value_counts()
pos_pct    = info['position'].value_counts(normalize=True) * 100

pos_df = pd.DataFrame({
    '影像數':   pos_counts,
    '比例 (%)': pos_pct.round(1),
})
pos_df.index = pos_df.index.map({'left': '左葉', 'right': '右葉'})
pos_df.index.name = '位置'

print('【位置分布】')
display(pos_df)

# ── View 分布 ──────────────────────────────────────────────────
view_counts = info['view'].value_counts()
view_pct    = info['view'].value_counts(normalize=True) * 100

view_df = pd.DataFrame({
    '影像數':   view_counts,
    '比例 (%)': view_pct.round(1),
})
view_df.index.name = 'View'

print('\n【View 分布】')
display(view_df)

# ── 位置 × View 交叉表 ─────────────────────────────────────────
cross_pv = pd.crosstab(
    info['position'].map({'left': '左葉', 'right': '右葉'}),
    info['view'],
    margins=True, margins_name='合計'
)
cross_pv.index.name = '位置'

print('\n【位置 × View 交叉表】')
display(cross_pv)

【位置分布】


,影像數,比例 (%)
位置,,
右葉,745,51.6
左葉,699,48.4



【View 分布】


,影像數,比例 (%)
View,,
saggital,739,51.2
transverse,705,48.8



【位置 × View 交叉表】


view,saggital,transverse,合計
位置,,,
右葉,389,356,745
左葉,350,349,699
合計,739,705,1444


In [94]:
info_all

,patient_ID,machine_type,file_name,position,view,Unnamed: 5,saggital,transverse
0,0A0BCFA9D26887AEA0EDD5B4815502D49453D70F,A,54125b513429422df1e6e43188b93e05,right,transverse,NaN,I,一
1,NaN,A,ee21cf68aab19f85ce00644e389c3183,right,saggital,NaN,NaN,NaN
2,0A5F1E8E321F99663A10CB28B828B416A858D8D9,B,65be383554977b95914d807bc609eda2_left,left,transverse,NaN,NaN,NaN
3,NaN,B,65be383554977b95914d807bc609eda2_right,left,saggital,NaN,NaN,NaN
4,0A56AC6CFDA6FE54384411A30C3112AD3F5EB324,C,e06eb8de4ebda1cffc48b94f9c502056,left,saggital,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
7338,D5DC5A162F746079E965D37AEC355559939D9A29,E,27ea54e2a7528a4a10054e14c33ded4a_left,right,saggital,NaN,NaN,NaN
7339,NaN,NaN,27ea54e2a7528a4a10054e14c33ded4a_right,right,transverse,NaN,NaN,NaN
7340,D992D29551C748785A9793C694C10AB36E962631,L,4dac657c11b83ded05d1687e55b84793,left,saggital,NaN,NaN,NaN
7341,D90982D9CD53E4000ACE0809A1DA0DC974660160,E,b7a72ac87eb64151c2863f309191124a_left,right,saggital,NaN,NaN,NaN


## 3. 回音特性分層（從 ABSTRACT 關鍵字萃取）

> 覆蓋率有限（僅部分報告有文字描述），以下為有描述的子集分析。

In [97]:
BASE = '../thyroid_old/data/CG_data/all_reports'
dfs_r = []
for f in sorted(os.listdir(BASE)):
    if not f.endswith('.xlsx'):
        continue
    path = os.path.join(BASE, f)
    try:
        df = pd.read_excel(path, engine='calamine')
    except Exception:
        df = pd.read_excel(path, engine='openpyxl')
    dfs_r.append(df)
all_rep = pd.concat(dfs_r, ignore_index=True)

# 只取 CG test 病人
cg_rep = all_rep[all_rep['IDCODE'].isin(cg_test_patients)][
    ['IDCODE', 'ABSTRACT', 'REPORT01', 'positive']
].copy()

# 從 ABSTRACT 及 REPORT01 萃取回音特性關鍵字
text_col = cg_rep['ABSTRACT'].fillna('') + ' ' + cg_rep['REPORT01'].fillna('')

echo_patterns = {
    'hypoechoic':       r'hypo.?echo',
    'hyperechoic':      r'hyper.?echo',
    'isoechoic':        r'iso.?echo',
    'cystic/mixed':     r'cystic|mixed echo',
    'calcification':    r'calcif',
    'ill-defined':      r'ill.?defined',
    'taller-than-wide': r'taller.than.wide|taller than wide',
    'halo':             r'\bhalo\b',
}

for label, pattern in echo_patterns.items():
    cg_rep[label] = text_col.str.contains(pattern, case=False, na=False).astype(int)

echo_cols = list(echo_patterns.keys())
echo_summary = pd.DataFrame({
    '特性':       echo_cols,
    '報告數':     [cg_rep[c].sum() for c in echo_cols],
    '覆蓋率 (%)': [(cg_rep[c].sum() / len(cg_rep) * 100) for c in echo_cols],
}).set_index('特性').round(1)

print(f'CG test 病人病歷報告數: {len(cg_rep)}\n')
display(echo_summary)

CG test 病人病歷報告數: 2737



,報告數,覆蓋率 (%)
特性,,
hypoechoic,17,0.6
hyperechoic,0,0.0
isoechoic,3,0.1
cystic/mixed,9,0.3
calcification,7,0.3
ill-defined,11,0.4
taller-than-wide,3,0.1
halo,3,0.1


## 4. 惡性程度分層（`positive` 欄：0 = 良性 / 1 = 惡性）

In [98]:
pos_map = {0.0: '良性 (0)', 1.0: '惡性 (1)'}
cg_rep['malignancy'] = cg_rep['positive'].map(pos_map)

mal_counts = cg_rep['malignancy'].value_counts(dropna=False)
mal_pct = cg_rep['malignancy'].value_counts(normalize=True, dropna=False) * 100

mal_df = pd.DataFrame({
    '報告數': mal_counts,
    '比例 (%)': mal_pct.round(1),
})
mal_df.index.name = '惡性程度'

print(f'有 positive 標記筆數: {cg_rep["positive"].notna().sum()} / {len(cg_rep)}\n')
display(mal_df)

# 各子群大小分層 × 惡性程度（僅有 positive 標記者）
print('\n--- 有標記病例中回音特性分布 ---')
labeled = cg_rep[cg_rep['positive'].notna()].copy()
labeled['malignancy'] = labeled['positive'].map({0.0: '良性', 1.0: '惡性'})
echo_by_mal = labeled.groupby('malignancy')[echo_cols].sum()
display(echo_by_mal)

有 positive 標記筆數: 2208 / 2737



,報告數,比例 (%)
惡性程度,,
良性 (0),1242,45.4
惡性 (1),966,35.3
NaN,529,19.3



--- 有標記病例中回音特性分布 ---


,hypoechoic,hyperechoic,isoechoic,cystic/mixed,calcification,ill-defined,taller-than-wide,halo
malignancy,,,,,,,,
惡性,0,0,0,0,0,0,0,0
良性,0,0,0,0,0,0,0,0


## 5. 綜合：位置 × 大小 × Single/Multiple 交叉表

In [ ]:
# 用 data_info_nodule 的 position 取代 filename 猜測
if 'pos_info' not in nodule_df.columns:
    nodule_df = nodule_df.merge(
        info[['file_name', 'position', 'view']].rename(columns={'position': 'pos_info', 'view': 'view_info'}),
        on='file_name', how='left'
    )
nodule_df['pos_info'] = nodule_df['pos_info'].str.strip().replace('lright', 'right')
nodule_df['loc_label'] = nodule_df['pos_info'].map({'left': '左葉', 'right': '右葉'}).fillna('不明')

# 只顯示有 mm 大小資料的結節
nodule_cross = nodule_df[nodule_df['size_group'].notna()].copy()

cross = pd.crosstab(
    index=[nodule_cross['loc_label'], nodule_cross['size_group']],
    columns=nodule_cross['nodule_type'],
    margins=True, margins_name='合計'
)
cross.index.names = ['位置', '大小（mm）']
display(cross)